In [ ]:
import sys; sys.path.append('..')
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX

from src.backtest import expanding_window_splits
from src.metrics import mase, seasonal_naive_scale, seasonal_naive_pred

model_df = pd.read_parquet('../data/model_df.parquet')

In [ ]:
# Verify lag_7 really equals sales from 7 days earlier
# rolling mean uses only prior days

one = sales_feat[sales_feat['id']== sales_feat['id'].iloc[0]].head(20)
one[['date', 'sales', 'lag_7', 'rolling_mean_7', 'rolling_std_7']]

In [ ]:
feature_cols = (
    [f'lag_{l}' for l in (7, 14, 28)]
    + [f'rolling_mean_{w}' for w in (7, 28)]
    + [f'rolling_std_{w}' for w in (7, 28)]
    + ['wday', 'month', 'is_event', 'snap', 'day', 'sell_price']
)

before = len(sales_feat)

#Drop rows with NaN features (first 28 days of each series, plus any missing prices)
model_df = sales_feat.dropna(subset=feature_cols).reset_index(drop=True)
print(f"Dropped {before - len(model_df)} rows with NaN features; {len(model_df)} remain.")
print(f"Feature columns: {feature_cols}")

In [ ]:
for i, (train_idx, test_idx) in enumerate(expanding_window_splits(model_df)):
    tr, te = model_df.loc[train_idx], model_df.loc[test_idx]
    print(f"Fold {i}:")
    print(f"  train {tr['date'].min().date()} → {tr['date'].max().date()}  ({len(tr):,} rows)")
    print(f"  test  {te['date'].min().date()} → {te['date'].max().date()}  ({len(te):,} rows)")
    print()

In [ ]:
for i, (train_idx, test_idx) in enumerate(expanding_window_splits(model_df)):
    tr, te = model_df.loc[train_idx], model_df.loc[test_idx]
    scale = seasonal_naive_scale(tr)
    score = mase(te['sales'].values, seasonal_naive_pred(te).values, scale)
    print(f"Fold {i}: seasonal-naive MASE = {score:.3f}")

In [ ]:
import warnings 
from statsmodels.tsa.statespace.sarimax import SARIMAX
warnings.filterwarnings("ignore")  # statsmodels is noisy

# Picking the 3 highest-volume series to keep the example small and fast. You can try more series, but it will take longer.
top_ids = (model_df.groupby('id')['sales'].sum()
           .sort_values(ascending=False).head(3).index.tolist())

Horizon = 28
results = []

for sid in top_ids:
    s = model_df[model_df['id'] == sid].sort_values('date')
    y = s['sales'].values
    train, test = y[:-Horizon], y[-Horizon:]

    # simple weekly-seasonal spec
    model = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,7),
                    enforce_stationarity=False, enforce_invertibility=False)
    fit = model.fit(disp=False)
    fcst = fit.forecast(steps=Horizon)

    #MASE using the per-series seasonal-naive scale on this series
    scale = np.mean(np.abs(train[7:] - train[:-7]))
    score = np.mean(np.abs(test - fcst)) / scale if scale != 0 else np.nan
    results.append((sid, score))
    print(f"{sid}: SARIMA MASE = {score: .3f}")

print("Per-series comparison (same series, same scaling):")
for sid in top_ids:
    s = model_df[model_df['id'] == sid].sort_values('date')
    y = s['sales'].values
    train, test = y[:-Horizon], y[-Horizon:]
    scale = np.mean(np.abs(train[7:] - train[:-7]))

    # seasonal-naive forecast for the 28-day test window = value 7 days earlier
    naive_fcst = y[-Horizon-7:-7]
    naive_mase = np.mean(np.abs(test - naive_fcst)) / scale
    print(f"{sid}: seasonal-naive MASE = {naive_mase:.3f}")

print("\nMean SARIMA MASE:", np.nanmean([r[1] for r in results]))   

In [ ]:
import lightgbm as lgb
import xgboost as xgb

# --- Feature setup ---
# Categorical features LightGBM should treat as such
cat_features = ['dept_id', 'wday', 'month']
# Make sure categoricals are 'category' dtype
for c in cat_features:
    model_df[c] = model_df[c].astype('category')

# Full feature list: your engineered features + categoricals
features = (
    ['lag_7', 'lag_14', 'lag_28',
     'rolling_mean_7', 'rolling_std_7', 'rolling_mean_28', 'rolling_std_28',
     'day', 'is_event', 'snap', 'sell_price']
    + cat_features
)
target = 'sales'

In [ ]:
def run_backtest(model_name):
    """Train one model type across the expanding-window folds, return MASE per fold."""
    scores = []
    for i, (train_idx, test_idx) in enumerate(expanding_window_splits(model_df)):
        tr, te = model_df.loc[train_idx], model_df.loc[test_idx]
        X_tr, y_tr = tr[features], tr[target]
        X_te, y_te = te[features], te[target]

        if model_name == 'lightgbm':
            model = lgb.LGBMRegressor(
                n_estimators=300, learning_rate=0.05,
                num_leaves=63, n_jobs=-1, verbose=-1
            )
            model.fit(X_tr, y_tr, categorical_feature=cat_features)
        else:  # xgboost
            model = xgb.XGBRegressor(
                n_estimators=300, learning_rate=0.05,
                max_depth=6, n_jobs=-1, enable_categorical=True
            )
            model.fit(X_tr, y_tr)

        pred = model.predict(X_te)
        pred = np.clip(pred, 0, None)  # sales can't be negative

        scale = seasonal_naive_scale(tr)        # same scale as the baseline
        score = np.mean(np.abs(y_te.values - pred)) / scale
        scores.append(score)
        print(f"  Fold {i}: MASE = {score:.3f}")
    print(f"  Mean MASE = {np.mean(scores):.3f}\n")
    return scores

print("LightGBM:")
lgbm_scores = run_backtest('lightgbm')

print("XGBoost:")
xgb_scores  = run_backtest('xgboost')

In [ ]:
print(f"{'Model':<22}{'Mean MASE':>10}")
print("-" * 32)
print(f"{'Seasonal-naive (all)':<22}{0.908:>10.3f}")   
print(f"{'LightGBM (global)':<22}{np.mean(lgbm_scores):>10.3f}")
print(f"{'XGBoost (global)':<22}{np.mean(xgb_scores):>10.3f}")